[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session4/student/Lab3_Radiology_Summarization_Student.ipynb)

# Lab 3: Fine-Tuning for Radiology Report Summarization
## CCI Session 4 — Cancer Care Informatics · KHCC AI Office

---

## Before You Start
**Enable GPU:** Go to `Runtime → Change runtime type → T4 GPU`

---

## Clinical Scenario

Every radiology report at KHCC has two sections:

- **Findings** — the detailed technical description of what the radiologist observed. Can be 100–300 words. Written for completeness.
- **Impression** — the 1–3 sentence clinical conclusion. The answer to "what does this mean for the patient?"

Writing the impression requires synthesizing findings into a concise clinical judgment. In a busy center reviewing 200+ chest X-rays per day, automating impression generation — even partially — could meaningfully reduce cognitive load.

In this lab, you will:
1. Load a real dataset of radiology reports (MIMIC-CXR)
2. Test a general-purpose language model *before* any training (baseline)
3. Fine-tune the model on radiology-specific data
4. Compare before vs. after using ROUGE metrics

---

## Key Concept: What Is Fine-Tuning?

A pre-trained language model like `flan-t5-small` has already learned from billions of words of general text. It knows English grammar, common medical terms, and summarization patterns — but it has never specifically learned that **radiology findings lead to radiology impressions**.

**Fine-tuning** continues training the model on your specific task. Instead of learning from scratch (which requires massive data and compute), you gently adjust the model's existing knowledge to fit your domain.

Think of it as a residency rotation after medical school: the doctor already has foundational knowledge, and the rotation specializes them for a specific clinical context.

```
Pre-trained model (general English)  →  Fine-tuned model (radiology summarization)
        flan-t5-small                →  flan-t5-small + your radiology data
```

---
## Cell 0: Setup — Install Libraries and Verify GPU

In [ ]:
# Install required libraries:
# - transformers : Hugging Face library for pre-trained models
# - datasets     : Hugging Face library for loading datasets
# - evaluate     : Hugging Face library for computing metrics (ROUGE, BLEU, etc.)
# - rouge_score  : Backend that `evaluate` uses to compute ROUGE
# - accelerate   : Speeds up training on GPU hardware
!pip install -q transformers datasets evaluate rouge_score accelerate

import torch

print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU model:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device:  {device}")

if not torch.cuda.is_available():
    print("\nWARNING: No GPU detected. Training will be very slow.")
    print("   Go to Runtime → Change runtime type → Select T4 GPU")

---
## Cell 1: Load the Radiology Dataset

We use **MIMIC-CXR** — a de-identified dataset of chest X-ray reports from Beth Israel Deaconess Medical Center (Boston, USA). It contains ~227,000 reports, each with a `findings` field and an `impression` field.

For training efficiency, we take a **random subset of 1,000 reports**. This is small by ML standards, but enough to demonstrate the fine-tuning effect within a lab session.

> **Why 1,000?** Training on 1,000 examples takes ~5–8 minutes on a T4 GPU. The full dataset would take hours. In a real deployment at KHCC, you would use all available data — ideally combined with your own institutional reports.

In [ ]:
# === CELL 1: LOAD RADIOLOGY REPORTS ===
from datasets import load_dataset

# TODO: Load the MIMIC-CXR summarization dataset from Hugging Face Hub
# ds = load_dataset("tgrex6/mimic-cxr-reports-summarization")
# print(ds)

# TODO: Preview 3 reports — print findings and impression side by side
# Note the pattern: findings (detailed technical text) -> impression (concise conclusion)
# for i in range(3):
#     print(f"--- Report {i+1} ---")
#     print(f"FINDINGS (input):    {ds['train'][i]['findings'][:250]}...")
#     print(f"IMPRESSION (target): {ds['train'][i]['impression']}")
#     print()

# TODO: Shuffle and take 1,000 examples for this lab
# seed=42 ensures everyone gets the same subset (reproducibility)
# small_ds = ds["train"].shuffle(seed=42).select(range(1000))
# print(f"Training subset size: {len(small_ds)} reports")

---
## Cell 2: Explore the Data

Before training any model, always understand your data. Key questions:
- How long are the inputs (findings) vs. the outputs (impressions)?
- Are there empty or corrupt records that could break training?
- What is the compression ratio — how much does the impression shorten the findings?

This informs the `max_length` parameters you will set during tokenization.

In [ ]:
# === CELL 2: DATA EXPLORATION ===

# TODO: Calculate average lengths (in words) of findings and impressions
# findings_lengths   = [len(x["findings"].split())   for x in small_ds]
# impression_lengths = [len(x["impression"].split()) for x in small_ds]
# avg_findings   = sum(findings_lengths) / len(findings_lengths)
# avg_impression = sum(impression_lengths) / len(impression_lengths)
# compression    = avg_findings / avg_impression
#
# print(f"Avg findings length:   {avg_findings:.0f} words")
# print(f"Avg impression length: {avg_impression:.0f} words")
# print(f"Compression ratio:     {compression:.1f}x")

# TODO: Check for empty records — these cause NaN loss during training
# empty_findings    = sum(1 for x in small_ds if not x["findings"].strip())
# empty_impressions = sum(1 for x in small_ds if not x["impression"].strip())
# print(f"Empty findings: {empty_findings}, Empty impressions: {empty_impressions}")

# TODO: Filter out any empty rows
# small_ds = small_ds.filter(
#     lambda x: len(x["findings"].strip()) > 0 and len(x["impression"].strip()) > 0
# )
# print(f"Clean dataset size: {len(small_ds)} reports")

---
## Cell 3: Load the Model and Tokenizer

We use **`google/flan-t5-small`** — a sequence-to-sequence (seq2seq) model trained by Google on 1,800+ language tasks with natural language instructions.

### Why flan-t5-small?

| Property | Value |
|---|---|
| Parameters | ~80 million |
| Architecture | Encoder-Decoder (T5) |
| Training | Instruction fine-tuned on diverse NLP tasks |
| Size on disk | ~308 MB |
| Why suitable | Designed for text-to-text tasks; responds well to prompts like "Summarize the following..." |

### Tokenizer vs. Model

- **Tokenizer**: Converts text → token IDs (numbers). Every model has its own vocabulary (~32,000 tokens for flan-t5).
- **Model**: Takes token IDs as input and produces output token IDs.

Both must always come from the same checkpoint to be compatible.

### Why `force_download=True`?

This flag bypasses the local Hugging Face cache and downloads fresh weights from the Hub. We use it to avoid accidentally loading a previously saved (possibly corrupted or overfit) checkpoint. Without fresh weights, the model loss can start at 0.0 and never learn anything meaningful.

### Why `dtype=torch.float32` and NOT `fp16`?

The flan-t5-small checkpoint has tied embedding weights (`shared.weight` and `lm_head.weight`) that cause numerical instability in float16, producing NaN loss from the very first step. Always use float32 with this checkpoint.

In [ ]:
# === CELL 3: LOAD MODEL AND TOKENIZER ===
import os, shutil
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Clean up any previous checkpoint folders from prior runs
for folder in ["./radiology-summarizer", "./radiology-summarizer-final"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Cleaned up: {folder}")

model_name = "google/flan-t5-small"

# TODO: Load tokenizer with force_download=True
# tokenizer = AutoTokenizer.from_pretrained(model_name, force_download=True)

# TODO: Load model with force_download=True and dtype=torch.float32
# IMPORTANT: Use dtype=torch.float32 (NOT fp16) to avoid NaN loss
# model = AutoModelForSeq2SeqLM.from_pretrained(
#     model_name,
#     force_download=True,
#     dtype=torch.float32,
# ).to(device)

# TODO: Print model info
# print(f"Model loaded on: {device}")
# print(f"Model parameters: {model.num_parameters():,}")

# TODO: Sanity check — verify the model produces real (non-NaN) logits
# model.eval()
# dummy = torch.ones(1, 10, dtype=torch.long).to(device)
# with torch.no_grad():
#     out = model(input_ids=dummy, decoder_input_ids=dummy)
# has_nan = out.logits.isnan().any().item()
# print(f"Logits contain NaN: {has_nan}")
# if not has_nan:
#     print("Model weights are healthy. Proceed.")

---
## Cell 4: Baseline — How Does the Model Perform *Before* Training?

Before fine-tuning, we test the model on 5 radiology reports to establish a **baseline**. This step is critical — without a baseline, you cannot measure whether fine-tuning helped.

### What to Expect at Baseline

The model has never seen radiology reports in a structured task-specific way. It will attempt to summarize using general language patterns, but typically:
- Uses generic, non-clinical phrases ("Observations are indicated")
- Misses specific findings (effusions, opacities, vertebral fractures)
- Sometimes hallucinates non-existent medical terms

### The Prompt Prefix

We prefix every input with `"Summarize the following radiology findings:\n"`. This instructs flan-t5 to treat the text as something to summarize rather than simply continuing it.

In [ ]:
# === CELL 4: BASELINE — BEFORE FINE-TUNING ===
import evaluate

rouge = evaluate.load("rouge")

PREFIX = "Summarize the following radiology findings:\n"

# Use the first 5 reports for a consistent before/after comparison
test_samples = small_ds.select(range(5))
actual_impressions = [s["impression"] for s in test_samples]

# TODO: Loop through the 5 test reports and generate predictions
# baseline_predictions = []
# model.eval()
#
# for i, sample in enumerate(test_samples):
#     input_text = PREFIX + sample["findings"]
#
#     # Tokenize the input
#     inputs = tokenizer(
#         input_text,
#         return_tensors="pt",
#         max_length=512,
#         truncation=True
#     ).to(device)
#
#     # Generate output
#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=128,
#             num_beams=4,
#             early_stopping=True
#         )
#
#     prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     baseline_predictions.append(prediction)
#
#     print(f"--- Report {i+1} ---")
#     print(f"ACTUAL:    {sample['impression'][:220]}")
#     print(f"PREDICTED: {prediction[:220]}")
#     print()

# TODO: Compute ROUGE scores for the baseline
# baseline_results = rouge.compute(
#     predictions=baseline_predictions,
#     references=actual_impressions
# )
# print("=== BASELINE ROUGE SCORES ===")
# for key, value in baseline_results.items():
#     print(f"{key}: {value:.4f}")
# print("\nSave these numbers — you will compare them after fine-tuning.")

---
## Cell 5: Preprocessing — Tokenize the Dataset for Training

Before training, we convert all text into token IDs and format them correctly for the seq2seq model.

### Key Concepts

#### `max_length` for inputs vs. labels
- **512 tokens for findings (input)**: Covers most radiology reports. Longer reports are truncated.
- **128 tokens for impressions (labels)**: Most impressions are 20–60 words. 128 is sufficient.

#### `padding="max_length"`
Pads shorter sequences with a special `[PAD]` token so all examples in a batch are the same length. Batching is essential for GPU efficiency.

#### The `-100` label masking trick (CRITICAL)
After padding, the label tensor contains real token IDs mixed with padding token IDs. We replace all padding positions with `-100`. Why?

PyTorch's cross-entropy loss function has a built-in `ignore_index=-100` parameter. Any position set to `-100` is **excluded from the loss calculation**. Without this, the model either:
- Produces NaN loss (undefined gradient), or
- Collapses to 0.0 loss (learns to predict padding everywhere)

#### Train/Validation Split
We split 85%/15% (850 train, 150 validation). The validation set is a held-out check — if validation loss starts rising while training loss keeps falling, the model is overfitting.

In [ ]:
# === CELL 5: PREPROCESSING ===

# TODO: Define the preprocessing function
# def preprocess_function(examples):
#     # Step 1: Add the task prefix to every findings text
#     inputs = [PREFIX + finding for finding in examples["findings"]]
#
#     # Step 2: Tokenize the inputs (findings → encoder input)
#     model_inputs = tokenizer(
#         inputs,
#         max_length=512,
#         truncation=True,
#         padding="max_length",
#     )
#
#     # Step 3: Tokenize the labels (impressions → decoder target)
#     labels = tokenizer(
#         examples["impression"],
#         max_length=128,
#         truncation=True,
#         padding="max_length",
#     )
#
#     # Step 4: CRITICAL — Replace padding token IDs with -100
#     label_ids = labels["input_ids"]
#     label_ids = [
#         [(token if token != tokenizer.pad_token_id else -100) for token in label]
#         for label in label_ids
#     ]
#     model_inputs["labels"] = label_ids
#     return model_inputs

# TODO: Split into train/validation (85%/15%)
# split = small_ds.train_test_split(test_size=0.15, seed=42)
# train_data = split["train"]
# val_data   = split["test"]
# print(f"Train: {len(train_data)}, Validation: {len(val_data)}")

# TODO: Apply tokenization to both splits
# tokenized_train = train_data.map(
#     preprocess_function, batched=True, remove_columns=train_data.column_names
# )
# tokenized_val = val_data.map(
#     preprocess_function, batched=True, remove_columns=val_data.column_names
# )

# TODO: Convert to PyTorch tensor format
# tokenized_train.set_format("torch")
# tokenized_val.set_format("torch")

# TODO: Verify shapes and -100 masking
# print(f"Input shape:  {tokenized_train[0]['input_ids'].shape}")
# print(f"Label shape:  {tokenized_train[0]['labels'].shape}")
# sample_labels = tokenized_train[0]['labels']
# n_real = (sample_labels != -100).sum().item()
# n_pad  = (sample_labels == -100).sum().item()
# print(f"Real tokens: {n_real}, Pad tokens (ignored): {n_pad}")

---
## Cell 6: Sanity Check — Verify Loss Before Training

**Always run this before a full training run.**

A freshly loaded model should have a high loss (typically 5–10) because it is guessing randomly among 32,000 vocabulary tokens. If loss is:

- **0.0** → weights are from a previously overfit checkpoint → re-run Cell 3
- **NaN** → weight corruption, dtype mismatch, or all labels are -100
- **5–10** → healthy starting point, proceed with training

In [ ]:
# === CELL 6: SANITY CHECK ===
from torch.utils.data import DataLoader

# TODO: Load one mini-batch and run a forward pass to check the loss
# loader = DataLoader(tokenized_train, batch_size=4, shuffle=True)
# batch = next(iter(loader))
# batch = {k: v.to(device) for k, v in batch.items()}
#
# model.train()
# with torch.no_grad():
#     outputs = model(**batch)
#
# loss_val = outputs.loss.item()
# print(f"Initial loss: {loss_val:.4f}")
#
# if loss_val == 0.0:
#     print("STOP: Loss is 0. Model loaded from overfit checkpoint. Re-run Cell 3.")
# elif loss_val != loss_val:  # NaN check
#     print("STOP: Loss is NaN. Restart runtime and re-run from top.")
# elif loss_val >= 3.0:
#     print(f"Loss looks healthy (~{loss_val:.1f}). Proceed to training.")
# else:
#     print("Loss is lower than expected for a fresh model. Verify weights.")

---
## Cell 7: Training Configuration — Hyperparameters Explained

**Hyperparameters** are settings you choose before training. Unlike model weights (learned automatically), these are set by the practitioner.

### Every Hyperparameter Explained

| Hyperparameter | Value | Why |
|---|---|---|
| `num_train_epochs=5` | 5 passes through all data | Too few → underfitting; too many → overfitting |
| `per_device_train_batch_size=8` | 8 examples per step | Fits in T4 GPU memory with 512-token inputs |
| `learning_rate=3e-5` | Conservative for fine-tuning | Too high (1e-3) → loss collapses to 0; too low (1e-8) → no progress |
| `weight_decay=0.05` | L2 regularization | Prevents overfitting, important with only ~850 examples |
| `warmup_steps=50` | Gradually ramp up LR | Prevents unstable updates in first steps |
| `eval_strategy="epoch"` | Evaluate after each epoch | Monitor validation loss for overfitting |
| `load_best_model_at_end=True` | Auto-select best checkpoint | Uses the epoch with lowest val loss |
| `fp16=False` | Full precision | flan-t5-small causes NaN with fp16 due to tied weights |

**Key diagnostic:** If loss = 0 at step 1, reduce learning rate by 10x.

In [ ]:
# === CELL 7: TRAINING CONFIGURATION ===
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# TODO: Define training arguments
# training_args = TrainingArguments(
#     output_dir="./radiology-summarizer-final",
#     num_train_epochs=5,
#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,
#     learning_rate=3e-5,
#     weight_decay=0.05,
#     warmup_steps=50,
#     logging_steps=25,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     fp16=False,              # MUST be False for flan-t5-small
#     report_to="none",
# )

# TODO: Create the data collator
# data_collator = DataCollatorForSeq2Seq(
#     tokenizer,
#     model=model,
#     pad_to_multiple_of=8
# )

# TODO: Create the Trainer
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_train,
#     eval_dataset=tokenized_val,
#     data_collator=data_collator,
# )

# TODO: Print the training plan
# steps_per_epoch = len(tokenized_train) // training_args.per_device_train_batch_size
# total_steps = steps_per_epoch * training_args.num_train_epochs
# print(f"Training examples: {len(tokenized_train)}")
# print(f"Steps per epoch:   {steps_per_epoch}")
# print(f"Total steps:       {total_steps}")
# print(f"Learning rate:     {training_args.learning_rate}")
# print("Trainer configured. Run the next cell to start training.")

---
## Cell 8: Fine-Tune the Model

This is the core training step. On a T4 GPU, expect **5–8 minutes** for 5 epochs on ~850 examples.

### What to Watch

**Healthy training:**
- Both training and validation loss decrease across epochs
- Validation loss stays close to training loss (no overfitting)
- Loss never hits 0.0 from step 1

**Warning signs:**

| Pattern | Likely Cause |
|---|---|
| Loss = 0.0 at step 1 | Broken/overfit weights — re-run Cell 3 |
| Validation loss = NaN | Label -100 masking not applied — re-run Cell 5 |
| Val loss rises while train loss falls | Overfitting — reduce epochs or increase weight_decay |

In [ ]:
# === CELL 8: FINE-TUNE THE MODEL ===

# TODO: Start training and watch the loss decrease
# print("Starting fine-tuning...")
# print("Expected time: 5-8 minutes on T4 GPU")
#
# train_result = trainer.train()
#
# print(f"\nTraining complete!")
# print(f"Total time:  {train_result.metrics['train_runtime']:.0f} seconds")
# print(f"Final loss:  {train_result.metrics['train_loss']:.4f}")
# print(f"Throughput:  {train_result.metrics['train_samples_per_second']:.1f} samples/sec")

---
## Cell 9: After Fine-Tuning — Qualitative Comparison

Run the same 5 test reports through the fine-tuned model and compare three outputs side-by-side:
1. **ACTUAL** — what the radiologist wrote
2. **BEFORE TUNING** — what the baseline model generated
3. **AFTER TUNING** — what the fine-tuned model generates

Read carefully. Look for:
- Did hallucinated words disappear?
- Is the model now using proper radiology terminology?
- Does the structure resemble a real impression (declarative, concise)?

In [ ]:
# === CELL 9: AFTER FINE-TUNING ===

# TODO: Run the SAME 5 reports through the fine-tuned model
# finetuned_predictions = []
# model.eval()
#
# for i, sample in enumerate(test_samples):
#     input_text = PREFIX + sample["findings"]
#     inputs = tokenizer(
#         input_text, return_tensors="pt", max_length=512, truncation=True
#     ).to(device)
#
#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=128,
#             num_beams=4,
#             early_stopping=True
#         )
#
#     prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     finetuned_predictions.append(prediction)
#
#     print(f"--- Report {i+1} ---")
#     print(f"ACTUAL:        {sample['impression'][:220]}")
#     print(f"BEFORE TUNING: {baseline_predictions[i][:220]}")
#     print(f"AFTER TUNING:  {prediction[:220]}")
#     print()

# TODO: Compute ROUGE scores for the fine-tuned model
# finetuned_results = rouge.compute(
#     predictions=finetuned_predictions,
#     references=actual_impressions,
# )

---
## Cell 10: ROUGE Metrics — What Do They Actually Measure?

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation) measures word overlap between the model's output and a reference summary.

### The Four ROUGE Variants

**ROUGE-1** — Unigram (single word) overlap. *Does the model use the right vocabulary?*
```
Reference:  "pleural effusion and pulmonary edema"
Prediction: "small pleural effusion"
Matches: pleural, effusion → ROUGE-1 = 2/4 = 0.50
```

**ROUGE-2** — Bigram (two consecutive words) overlap. *Has the model learned domain-specific phrases?* This is the **most meaningful** metric for fine-tuning. Improvement here means the model learned radiology two-word phrases.

**ROUGE-L** — Longest Common Subsequence. *Is the sentence structure correct?*

**ROUGE-Lsum** — Sentence-level ROUGE-L. Better for multi-sentence impressions.

### Score Ranges

| ROUGE Score | Interpretation |
|---|---|
| 0.00–0.10 | Very poor — unrelated or hallucinated text |
| 0.10–0.25 | Poor — some vocabulary but wrong structure |
| 0.25–0.40 | Moderate — captures key findings, misses detail |
| 0.40–0.60 | Good — competitive with human summarization |
| > 0.60 | Excellent — approaches radiologist-level |

### Critical Limitations

1. **ROUGE measures overlap, not correctness.** "no pneumonia" and "pneumonia" both score on the word "pneumonia" — but clinically they are opposite.
2. **Synonyms are penalized.** "Fluid around the lung" and "pleural effusion" get 0 overlap despite meaning the same thing.
3. **ROUGE cannot replace radiologist evaluation.** Always involve clinicians in assessing model outputs.

In [ ]:
# === CELL 10: ROUGE COMPARISON ===

# TODO: Print a side-by-side comparison table
# print(f"{'Metric':<12} {'Baseline':>10} {'Fine-Tuned':>12} {'Change':>10}")
# print("-" * 58)
#
# for metric in ["rouge1", "rouge2", "rougeL", "rougeLsum"]:
#     base_val = baseline_results[metric]
#     ft_val   = finetuned_results[metric]
#     delta    = ft_val - base_val
#     arrow    = "UP" if delta > 0.001 else "DOWN" if delta < -0.001 else "="
#     print(f"{metric:<12} {base_val:>10.4f} {ft_val:>12.4f}   {arrow} {abs(delta):.4f}")
#
# print()
# print("Key question: Did ROUGE-2 improve?")
# print("If yes → the model learned radiology-specific phrase patterns.")
# print("If no  → more data or a larger model is needed.")

---
## Stretch Challenge: Tune the Hyperparameters

Try changing **one hyperparameter at a time** and observe the effect on ROUGE-2.

| Experiment | Change | Hypothesis |
|---|---|---|
| A | `learning_rate=1e-5` | Slower, more stable — may need more epochs |
| B | `learning_rate=1e-4` | Faster, higher risk of loss collapse — what happens at step 1? |
| C | `num_train_epochs=10` | More passes — does val loss keep improving or start rising? |
| D | `per_device_train_batch_size=4` | Noisier gradients — does it help or hurt? |
| E | `weight_decay=0.1` | More regularization — does it improve val loss on epoch 3+? |

**Important**: Always reload the base model before each experiment.

**Reflection:** Which hyperparameter had the biggest impact on ROUGE-2? Why?

In [ ]:
# === STRETCH CHALLENGE ===
# Uncomment and modify one hyperparameter per experiment.
# Always reload the base model first!

# from transformers import AutoModelForSeq2SeqLM
# model_exp = AutoModelForSeq2SeqLM.from_pretrained(
#     "google/flan-t5-small",
#     force_download=True,
#     dtype=torch.float32,
# ).to(device)
#
# training_args_exp = TrainingArguments(
#     output_dir="./experiment-A",
#     num_train_epochs=5,
#     per_device_train_batch_size=8,
#     learning_rate=1e-5,        # <- Change this per experiment
#     weight_decay=0.05,
#     warmup_steps=50,
#     logging_steps=25,
#     eval_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",
#     fp16=False,
#     report_to="none",
# )
#
# trainer_exp = Trainer(
#     model=model_exp,
#     args=training_args_exp,
#     train_dataset=tokenized_train,
#     eval_dataset=tokenized_val,
#     data_collator=DataCollatorForSeq2Seq(tokenizer, model=model_exp, pad_to_multiple_of=8),
# )
# trainer_exp.train()
# Then re-run prediction and ROUGE cells using model_exp

print("Uncomment the code above to run hyperparameter experiments.")
print("Change one variable at a time and compare ROUGE-2 results.")

---
## KHCC Clinical Context and Next Steps

### What This Lab Demonstrated

| Concept | What You Observed |
|---|---|
| Fine-tuning effect | Model shifted from generic/hallucinated text to coherent radiology language |
| Loss as a training signal | Healthy loss decreased from ~9 → ~7 across 5 epochs |
| Validation monitoring | Val loss tracked train loss — no overfitting with these settings |
| ROUGE as evaluation | Quantitative, reproducible comparison of before vs. after |

### Scaling to Production at KHCC

| This Lab | Production System |
|---|---|
| 850 training examples | Full MIMIC-CXR (93K) + KHCC archive |
| flan-t5-small (80M params) | BioMedLM, Llama-3-Med, or GPT-4 fine-tuned |
| ROUGE only | ROUGE + blinded radiologist rating study (n >= 50) |
| English only | Arabic/English bilingual |
| Standalone script | Integrated with RIS (Radiology Information System) |
| No safety review | IRB approval, clinical risk assessment, fail-safe logic |

### Practical Takeaways

1. **Domain-specific fine-tuning works**, even with limited data
2. **Debugging training requires systematic checks** — sanity check loss before training, monitor val loss during, inspect outputs after
3. **ROUGE is a starting point**, not a final evaluation — always involve clinicians
4. **Hyperparameters matter enormously** — a 10x too-high learning rate can completely prevent learning

---
*CCI Session 4 — KHCC AI Office · King Hussein Cancer Center, Amman, Jordan*